# Bell State Preparation experiment: Compute $\langle \bar{\Phi}^+ | \bar{Z}\bar{Z} | \bar{\Phi}^+\rangle$ with the seven-qubit Steane code

In [1]:
from typing import List, Dict, Sequence
import itertools
import functools
import numpy as np
from tqdm import tqdm
import cirq
import qiskit
from qiskit.circuit.library import Barrier
import qiskit_ibm_runtime
from qiskit_ibm_runtime import SamplerV2 as Sampler

import stim
import stimcirq

from encoded.diagonalize import get_stabilizer_matrix_from_paulis, get_measurement_circuit, get_paulis_from_stabilizer_matrix, string_to_cirq_paulistring
from encoded.tcc import tcc_encoding

/Users/ryan/prof/work/encoded/envencoded/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
import datetime


time_key = datetime.datetime.now().strftime("%m_%d_%Y_%H:%M:%S")  # For saving results.

## Set parameters

In [3]:
d = 3                                   # Code distance.
nshots = 10_000                         # Number of samples/shots
depth = 0                               # Number of folded Bell state preparation circuits for added noise
k = 2                                   # Number of logical qubits.

In [4]:
distance_to_nqubits = lambda d: (3 * d ** 2 + 1) // 4

n = distance_to_nqubits(d)

In [83]:
# Computer and qubits to use.
service = qiskit_ibm_runtime.QiskitRuntimeService(
    channel="ibm_cloud",
    token="Vwp-noJ99qjdtslYdRY6aEClfwr6PM63bP1Az7fE3015",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/8adb079e32e34c679acff94b68640033:f54da97d-7fe5-4108-9409-98a68f6ca53f::",
)
computer = service.backend("ibm_boston")
sampler = Sampler(computer)

# See calibration data at https://quantum.ibm.com/services/resources to select good qubits.
layout = {
    1: [2, 3],
    3: [29, 30, 31, 32, 33, 38, 39, 48, 49, 50, 51, 52, 53, 54],
    5: list(range(38)),
}


# The best qubits ever on Fez Feb 6-8.
# layout = {
#     1 : [123, 124],
#     7 : [123, 124, 125, 126, 127, 128, 136, 137, 142, 143, 144, 145, 146, 147],
# }


# Good on Kyiv 2/6
# layout = {
#     1 : [20, 21],
#     7 : [0, 1, 2, 3, 4, 5, 14, 15, 18, 19, 20, 21, 22, 23],
# }

# Good on Sherbrooke 2/5.
# layout = {
#     1 : [103, 104],
#     7 : [103, 104, 105, 106, 107, 108, 111, 112, 121, 122, 123, 124, 125, 126],
# }

# Good on Kyiv 2/5.
# layout = {
#     1 : [95, 96],
#     7 : [95, 96, 97, 98, 99, 100, 101, 113, 114, 115, 116, 117, 118, 119],
# }

qiskit_runtime_service._discover_account:WARNING:2026-04-12 16:29:11,043: Loading account with the given token. A saved account will not be used.


In [85]:
len(layout[d])

14

In [6]:
# Expectation of pauli on bitstring measured in diagonal basis.
def compute_expectation(
    pauli: cirq.PauliString,
    counts: Dict[str, int],
) -> float:
    if pauli is cirq.PauliString():
        return 1.0

    expectation = 0.0

    indices = [q.x for q in pauli.qubits]
    for key, value in counts.items():
        key = list(map(int, list(key[::-1])))
        expectation += (-1) ** sum([key[i] for i in indices]) * value

    return pauli.coefficient * expectation / sum(counts.values())

### Run unmitigated experiment

In [7]:
qreg = cirq.LineQubit.range(k)

circuit = cirq.Circuit()
circuit.append(cirq.H.on(qreg[0]))
for i in range(len(qreg)-1):
    circuit.append(cirq.CNOT.on(qreg[i], qreg[i+1]))

circuit = qiskit.QuantumCircuit.from_qasm_str(circuit.to_qasm())
circuit.measure_active()
# Compile to device.
compiled_physical = qiskit.transpile(
    circuit, 
    backend=computer,
    initial_layout=layout[1],  # Hardcode n = 1 (i.e., no encoding) to get layout.
    routing_method="sabre",
    # scheduling_method="asap",
    optimization_level=0,
)
print(compiled_physical.draw(fold=-1, idle_wires=False))

global phase: 3π/4
         ┌─────────┐┌────┐┌─────────┐                                ░ ┌─┐   
q_0 -> 2 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■──────────────────────────────░─┤M├───
         ├─────────┤├────┤├─────────┤ │ ┌─────────┐┌────┐┌─────────┐ ░ └╥┘┌─┐
q_1 -> 3 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─░──╫─┤M├
         └─────────┘└────┘└─────────┘   └─────────┘└────┘└─────────┘ ░  ║ └╥┘
 meas: 2/═══════════════════════════════════════════════════════════════╩══╩═
                                                                        0  1 


In [ ]:
job_physical = sampler.run(
    [compiled_physical],
    shots=nshots
)

In [ ]:
all_counts_raw = [result.data.meas.get_counts() for result in job_physical.result()]
all_counts_raw

In [ ]:
ev_physical = compute_expectation(string_to_cirq_paulistring("ZZ"), all_counts_raw[0])
print("Physical result:", ev_physical)

## Run encoded experiment

### Compute/load 0_L / 1_L bitstrings

In [16]:
try:
    zero_bitstrings = np.loadtxt(f"zero_bitstrings_d{d}.txt")
    print("Loaded zero bitstrings (ints) from memory:", zero_bitstrings)
except FileNotFoundError:
    print("Unable to load zero bitstrings from memory, computing them...")
    encode_circuit = tcc_encoding(d)
    encode_circuit = cirq.H.on_each(encode_circuit.all_qubits()) + encode_circuit[1:]  # Remove RX gates to prepare the |+> state in favor of H.
    s = stimcirq.cirq_circuit_to_stim_circuit(encode_circuit + cirq.measure(encode_circuit.all_qubits())).compile_sampler()
    zero_bitstrings = sorted(set([int(b[0]) for b in s.sample(1_000_000, bit_packed=True)]))
    print("Computed these bitstrings (ints) and saved to memory:", zero_bitstrings)
    np.savetxt(f"zero_bitstrings_d{d}.txt", zero_bitstrings)

try:
    one_bitstrings = np.loadtxt(f"one_bitstrings_d{d}.txt")
    print("Loaded one bitstrings (ints) from memory:", zero_bitstrings)
except FileNotFoundError:
    print("Unable to load one bitstrings from memory, computing them...")
    encode_circuit = tcc_encoding(d)
    encode_circuit = cirq.H.on_each(encode_circuit.all_qubits()) + encode_circuit[1:]  # Remove RX gates to prepare the |+> state in favor of H.
    encode_circuit.append(cirq.X.on_each(encode_circuit.all_qubits()))  # Put into 1_L state.
    s = stimcirq.cirq_circuit_to_stim_circuit(encode_circuit + cirq.measure(encode_circuit.all_qubits())).compile_sampler()
    one_bitstrings = sorted(set([int(b[0]) for b in s.sample(1_000_000, bit_packed=True)]))
    print("Computed these bitstrings (ints) and saved to memory:", one_bitstrings)
    np.savetxt(f"one_bitstrings_d{d}.txt", one_bitstrings)

Loaded zero bitstrings (ints) from memory: [  0.  15.  54.  57.  85.  90.  99. 108.]
Loaded one bitstrings (ints) from memory: [  0.  15.  54.  57.  85.  90.  99. 108.]


In [17]:
zero_bitstrings

array([  0.,  15.,  54.,  57.,  85.,  90.,  99., 108.])

In [18]:
one_bitstrings

array([ 19.,  28.,  37.,  42.,  70.,  73., 112., 127.])

### Unit tests

In [21]:
# Get circuit to encode |0_L 0_L>.
encode_circuit = tcc_encoding(d)  # Encodes one |0_L>.
encode_circuit = cirq.H.on_each(encode_circuit.all_qubits()) + encode_circuit[1:]  # Remove RX gates to prepare the |+> state in favor of H.

qreg = cirq.LineQubit.range(n * k)
encoding = cirq.Circuit.concat_ragged(
    encode_circuit,
    encode_circuit.transform_qubits({a: b for a, b in zip(qreg[:n], qreg[n:])})
)
# Test that preparing |0_L 0_L> always measures |0_L> on both qubits.
test = encoding.copy()
result = cirq.Simulator().run(test + cirq.measure(test.all_qubits(), key="z"), repetitions=10_000)
bitstrings = result.measurements.get("z")

for bitstring in bitstrings:
    q0 = int("".join(str(s) for s in bitstring[:n][::-1]), 2)
    q1 = int("".join(str(s) for s in bitstring[n:][::-1]), 2)
    assert q0 in zero_bitstrings
    assert q1 in zero_bitstrings

# Test that preparing |0_L 1_L> always measures |0_L> on the first qubit and |1_L> on the second qubit.
test = encoding.copy()
test.append(cirq.X.on_each(qreg[n:]))
result = cirq.Simulator().run(test + cirq.measure(test.all_qubits(), key="z"), repetitions=10_000)
bitstrings = result.measurements.get("z")

for bitstring in bitstrings:
    q0 = int("".join(str(s) for s in bitstring[:n][::-1]), 2)
    q1 = int("".join(str(s) for s in bitstring[n:][::-1]), 2)
    assert q0 in zero_bitstrings
    assert q1 in one_bitstrings
encoding.append(cirq.X.on_each(qreg[:n]))

# Test that preparing |+_L 0_L> measures |0_L> or |1_L> on the first qubit and |0_L> on the second qubit.
test = encoding.copy()
test.append(cirq.H.on_each(qreg[:n]))
result = cirq.Simulator().run(test + cirq.measure(test.all_qubits(), key="z"), repetitions=10_000)
bitstrings = result.measurements.get("z")

for bitstring in bitstrings:
    q0 = int("".join(str(s) for s in bitstring[:n][::-1]), 2)
    q1 = int("".join(str(s) for s in bitstring[n:][::-1]), 2)
    assert q0 in zero_bitstrings or q0 in one_bitstrings
    assert q1 in zero_bitstrings
encoding.append(cirq.X.on_each(qreg[:n]))

### Run experiment

In [30]:
encode_circuit = tcc_encoding(d)
qubits = sorted(encode_circuit.all_qubits())
encode_circuit = cirq.H.on_each(qubits) + encode_circuit[1:]  # Remove RX gates to prepare the |+> state in favor of H.

qreg = cirq.LineQubit.range(n * k)
encoding = cirq.Circuit.concat_ragged(
    encode_circuit,
    encode_circuit.transform_qubits({a: b for a, b in zip(qreg[:n], qreg[n:])})
)
encoding.append(cirq.X.on_each(qreg[:n]))

# Prepare Bell state
encoding.append(cirq.Moment(cirq.H.on_each(qreg[:n])))
encoding.append(cirq.CNOT.on_each([(qreg[i], qreg[i+n]) for i in range(n)]))
encoding

┌──┐   ┌──┐   ┌──┐                       ┌───────┐
0: ────H───@───@────@───────────────────────────H───X───H────@──────────
           │   │    │                                        │
1: ────H───┼───┼────┼@─────@────────────────────H───X───H────┼@─────────
           │   │    ││     │                                 ││
2: ────H───┼───┼────┼┼─────┼@─────@─────────────H───X───H────┼┼@────────
           │   │    ││     ││     │                          │││
3: ────H───@───┼────┼@─────┼@─────┼X────────────────────H────┼┼┼@───────
               │    │      │      │                          ││││
4: ────H───────@────┼──────┼──────@─────@───X───────────H────┼┼┼┼@──────
                    │      │            │                    │││││
5: ────H────────────@──────@────────────┼───@───X───────H────┼┼┼┼┼@─────
                                        │   │                ││││││
6: ────H────────────────────────────────@───@───H───X───H────┼┼┼┼┼┼@────
                                                             │││││││
7: ────H───@───@────@───────────────────────────H────────────X┼┼┼┼┼┼────
           │   │    │                                         ││││││
8: ────H───┼───┼────┼@─────@────────────────────H─────────────X┼┼┼┼┼────
           │   │    ││     │                                   │││││
9: ────H───┼───┼────┼┼─────┼@─────@─────────────H──────────────X┼┼┼┼────
           │   │    ││     ││     │                             ││││
10: ───H───@───┼────┼@─────┼@─────┼─────────────────────────────X┼┼┼────
               │    │      │      │                              │││
11: ───H───────@────┼──────┼──────@─────@────────────────────────X┼┼────
                    │      │            │                         ││
12: ───H────────────@──────@────────────┼───@─────────────────────X┼────
                                        │   │                      │
13: ───H────────────────────────────────@───@───H──────────────────X────
                   └──┘   └──┘   └──┘                       └───────┘

In [31]:
zero_bitstrings

array([  0.,  15.,  54.,  57.,  85.,  90.,  99., 108.])

In [32]:
one_bitstrings

array([ 19.,  28.,  37.,  42.,  70.,  73., 112., 127.])

In [52]:
result = cirq.Simulator().run(encoding + cirq.measure(encoding.all_qubits(), key="z"), repetitions=10_000)
bit_arrays = result.measurements.get("z")

for bitstring in bit_arrays:
    q0 = int("".join(str(s) for s in bitstring[:n][::-1]), 2)
    q1 = int("".join(str(s) for s in bitstring[n:][::-1]), 2)
    if q0 in zero_bitstrings:
        assert q1 in zero_bitstrings
    elif q0 in one_bitstrings:
        assert q1 in one_bitstrings
    else:
        raise AssertionError("Neither 0_L 0_L nor 1_L 1_L were measured.")

In [53]:
encoding = qiskit.QuantumCircuit.from_qasm_str(encoding.to_qasm())
encoding.draw(fold=-1)

┌───┐            ┌───┐┌───┐┌───┐                                                            
 q_0: ┤ H ├─■──■─────■─┤ H ├┤ X ├┤ H ├──────────────────────■─────────────────────────────────────
      ├───┤ │  │     │ └───┘└───┘├───┤┌───┐┌───┐            │                                     
 q_1: ┤ H ├─┼──┼──■──┼────────■──┤ H ├┤ X ├┤ H ├────────────┼─────────■───────────────────────────
      ├───┤ │  │  │  │        │  └───┘├───┤├───┤┌───┐       │         │                           
 q_2: ┤ H ├─┼──┼──┼──┼───■────┼────■──┤ H ├┤ X ├┤ H ├───────┼─────────┼────■──────────────────────
      ├───┤ │  │  │  │   │    │    │  ├───┤├───┤└───┘       │         │    │                      
 q_3: ┤ H ├─■──┼──■──┼───■────┼────┼──┤ X ├┤ H ├──■─────────┼─────────┼────┼──────────────────────
      ├───┤    │     │        │    │  └───┘├───┤  │  ┌───┐  │         │    │                      
 q_4: ┤ H ├────■─────┼────────┼────■────■──┤ X ├──┼──┤ H ├──┼─────────┼────┼────■─────────────────
      ├───┤          │        │         │  └───┘  │  ├───┤  │  ┌───┐  │    │    │                 
 q_5: ┤ H ├──────────■────────■─────────┼────■────┼──┤ X ├──┼──┤ H ├──┼────┼────┼─────────■───────
      ├───┤                             │    │    │  ├───┤  │  ├───┤  │    │    │  ┌───┐  │       
 q_6: ┤ H ├─────────────────────────────■────■────┼──┤ H ├──┼──┤ X ├──┼────┼────┼──┤ H ├──┼────■──
      ├───┤            ┌───┐                      │  └───┘┌─┴─┐└───┘  │    │    │  └───┘  │    │  
 q_7: ┤ H ├─■──■─────■─┤ H ├──────────────────────┼───────┤ X ├───────┼────┼────┼─────────┼────┼──
      ├───┤ │  │     │ └───┘     ┌───┐            │       └───┘     ┌─┴─┐  │    │         │    │  
 q_8: ┤ H ├─┼──┼──■──┼────────■──┤ H ├────────────┼─────────────────┤ X ├──┼────┼─────────┼────┼──
      ├───┤ │  │  │  │        │  └───┘┌───┐       │                 └───┘┌─┴─┐  │         │    │  
 q_9: ┤ H ├─┼──┼──┼──┼───■────┼────■──┤ H ├───────┼──────────────────────┤ X ├──┼─────────┼────┼──
      ├───┤ │  │  │  │   │    │    │  └───┘     ┌─┴─┐                    └───┘  │         │    │  
q_10: ┤ H ├─■──┼──■──┼───■────┼────┼────────────┤ X ├───────────────────────────┼─────────┼────┼──
      ├───┤    │     │        │    │            └───┘                         ┌─┴─┐       │    │  
q_11: ┤ H ├────■─────┼────────┼────■────■─────────────────────────────────────┤ X ├───────┼────┼──
      ├───┤          │        │         │                                     └───┘     ┌─┴─┐  │  
q_12: ┤ H ├──────────■────────■─────────┼────■──────────────────────────────────────────┤ X ├──┼──
      ├───┤                             │    │  ┌───┐                                   └───┘┌─┴─┐
q_13: ┤ H ├─────────────────────────────■────■──┤ H ├────────────────────────────────────────┤ X ├
      └───┘                                     └───┘                                        └───┘

In [55]:
to_run = encoding.copy()
to_run.barrier()

# Idle time/X gates.
for _ in range(depth):
    to_run.x(to_run.qubits)
    to_run.barrier()
    to_run.x(to_run.qubits)
    to_run.barrier()

to_run.measure_all()

to_run.draw(fold=-1)

┌───┐            ┌───┐┌───┐┌───┐                                                             ░  ░ ┌─┐                                       
    q_0: ┤ H ├─■──■─────■─┤ H ├┤ X ├┤ H ├──────────────────────■──────────────────────────────────────░──░─┤M├───────────────────────────────────────
         ├───┤ │  │     │ └───┘└───┘├───┤┌───┐┌───┐            │                                      ░  ░ └╥┘┌─┐                                    
    q_1: ┤ H ├─┼──┼──■──┼────────■──┤ H ├┤ X ├┤ H ├────────────┼─────────■────────────────────────────░──░──╫─┤M├────────────────────────────────────
         ├───┤ │  │  │  │        │  └───┘├───┤├───┤┌───┐       │         │                            ░  ░  ║ └╥┘┌─┐                                 
    q_2: ┤ H ├─┼──┼──┼──┼───■────┼────■──┤ H ├┤ X ├┤ H ├───────┼─────────┼────■───────────────────────░──░──╫──╫─┤M├─────────────────────────────────
         ├───┤ │  │  │  │   │    │    │  ├───┤├───┤└───┘       │         │    │                       ░  ░  ║  ║ └╥┘┌─┐                              
    q_3: ┤ H ├─■──┼──■──┼───■────┼────┼──┤ X ├┤ H ├──■─────────┼─────────┼────┼───────────────────────░──░──╫──╫──╫─┤M├──────────────────────────────
         ├───┤    │     │        │    │  └───┘├───┤  │  ┌───┐  │         │    │                       ░  ░  ║  ║  ║ └╥┘┌─┐                           
    q_4: ┤ H ├────■─────┼────────┼────■────■──┤ X ├──┼──┤ H ├──┼─────────┼────┼────■──────────────────░──░──╫──╫──╫──╫─┤M├───────────────────────────
         ├───┤          │        │         │  └───┘  │  ├───┤  │  ┌───┐  │    │    │                  ░  ░  ║  ║  ║  ║ └╥┘┌─┐                        
    q_5: ┤ H ├──────────■────────■─────────┼────■────┼──┤ X ├──┼──┤ H ├──┼────┼────┼─────────■────────░──░──╫──╫──╫──╫──╫─┤M├────────────────────────
         ├───┤                             │    │    │  ├───┤  │  ├───┤  │    │    │  ┌───┐  │        ░  ░  ║  ║  ║  ║  ║ └╥┘┌─┐                     
    q_6: ┤ H ├─────────────────────────────■────■────┼──┤ H ├──┼──┤ X ├──┼────┼────┼──┤ H ├──┼────■───░──░──╫──╫──╫──╫──╫──╫─┤M├─────────────────────
         ├───┤            ┌───┐                      │  └───┘┌─┴─┐└───┘  │    │    │  └───┘  │    │   ░  ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                  
    q_7: ┤ H ├─■──■─────■─┤ H ├──────────────────────┼───────┤ X ├───────┼────┼────┼─────────┼────┼───░──░──╫──╫──╫──╫──╫──╫──╫─┤M├──────────────────
         ├───┤ │  │     │ └───┘     ┌───┐            │       └───┘     ┌─┴─┐  │    │         │    │   ░  ░  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐               
    q_8: ┤ H ├─┼──┼──■──┼────────■──┤ H ├────────────┼─────────────────┤ X ├──┼────┼─────────┼────┼───░──░──╫──╫──╫──╫──╫──╫──╫──╫─┤M├───────────────
         ├───┤ │  │  │  │        │  └───┘┌───┐       │                 └───┘┌─┴─┐  │         │    │   ░  ░  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐            
    q_9: ┤ H ├─┼──┼──┼──┼───■────┼────■──┤ H ├───────┼──────────────────────┤ X ├──┼─────────┼────┼───░──░──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├────────────
         ├───┤ │  │  │  │   │    │    │  └───┘     ┌─┴─┐                    └───┘  │         │    │   ░  ░  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐         
   q_10: ┤ H ├─■──┼──■──┼───■────┼────┼────────────┤ X ├───────────────────────────┼─────────┼────┼───░──░──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├─────────
         ├───┤    │     │        │    │            └───┘                         ┌─┴─┐       │    │   ░  ░  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐      
   q_11: ┤ H ├────■─────┼────────┼────■────■─────────────────────────────────────┤ X ├───────┼────┼───░──░──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├──────
         ├───┤          │        │         │                                     └───┘     ┌─┴─┐  │   ░  ░  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐   
   q_12: ┤ H ├──────────■────────■─────────┼────■──────────────────────────────────────────┤ X ├──┼───░──░──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├───
         ├───┤                             │    │  ┌───┐                                   └───┘┌─┴─┐ ░  ░  ║

In [56]:
to_run = qiskit.transpile(
    to_run, 
    backend=computer,
    initial_layout=layout[d],
    routing_method="sabre",
    # scheduling_method="asap",
    optimization_level=3,
)
to_run.draw(fold=-1, idle_wires=False)

global phase: 5π/4
           ┌─────────┐┌────┐ ┌───────┐                                                                         ┌────┐┌─────────┐                                                                                                                                                         ┌────┐         ┌────────┐   ┌────┐ ┌────────┐         ┌────┐  ┌─────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ░  ░          ┌─┐                              
  q_0 -> 2 ┤ Rz(π/2) ├┤ √X ├─┤ Rz(π) ├────────────────────────────────────────────────────────────────■────────┤ √X ├┤ Rz(π/2) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■────┤ √X ├──■──────┤ Rz(-π) ├───┤ √X ├─┤ Rz(-π) ├───■─────┤ √X ├──┤ Rz(π/2) ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────░──░──────────┤M├──────────────────────────────
           └┬────────┤├────┤┌┴───────┴┐                                     ┌────┐   ┌────┐           │        └────┘└─────────┘                                                                                                                          ┌──────────┐  ┌────┐      │    ├────┤  │      └─┬────┬─┘   └────┘ └────────┘   │   ┌─┴────┴─┐└──┬────┬─┘         ┌────┐             ┌────────┐┌────┐  ┌────────┐      ┌────┐  ┌──────────┐                                                                                                                                ┌────┐                 ┌────┐      ┌────┐   ┌─────────────┐┌────┐           ┌────────┐   ┌────┐         ┌────┐┌─────────────┐┌────┐┌──────────┐                                           ┌────┐                       ┌────┐                   ┌─────────┐    ┌────┐  ┌──────────┐     │        ┌────┐   ┌─────────┐        ┌────┐           ┌────┐                                                                                                                                                                                                                                                                                                                                                                                   ░  ░    ┌─┐   └╥┘                              
  q_1 -> 3 ─

In [69]:
from qiskit_aer import AerSimulator

job = AerSimulator().run(to_run, shots=nshots)

In [70]:
res = job.result()
res.get_counts()

{'10101011011010': 72,
 '10101011010101': 70,
 '10001101111111': 72,
 '00000000111001': 79,
 '11100001000110': 78,
 '00100110101010': 70,
 '10110100111001': 88,
 '10101010001111': 74,
 '00011111010101': 94,
 '00111001110000': 85,
 '10010010100101': 75,
 '11000110110110': 77,
 '01110010111001': 80,
 '01110011011010': 73,
 '00100110010011': 90,
 '01110010000000': 67,
 '00111001001001': 76,
 '11111111000110': 62,
 '01101100000000': 85,
 '11011000001111': 71,
 '10101010000000': 91,
 '01001011111111': 65,
 '00011110111001': 74,
 '10110101100011': 65,
 '10010010010011': 76,
 '01001011001001': 72,
 '11100000100101': 84,
 '00100111110000': 72,
 '11111110100101': 93,
 '11011000110110': 79,
 '11011001101100': 74,
 '01010101001001': 74,
 '01101101100011': 71,
 '11111110101010': 85,
 '00000001011010': 72,
 '10010010011100': 85,
 '10110101101100': 94,
 '11000111100011': 74,
 '00011110000000': 83,
 '11111110011100': 74,
 '00011110110110': 69,
 '11111111110000': 83,
 '11100001001001': 87,
 '001001110

In [71]:
zero_bitstrings

array([  0.,  15.,  54.,  57.,  85.,  90.,  99., 108.])

In [72]:
one_bitstrings

array([ 19.,  28.,  37.,  42.,  70.,  73., 112., 127.])

In [77]:
bit_arrays = result.measurements.get("z")

for bitstring in res.get_counts().keys():
    q0 = int("".join(str(s) for s in bitstring[:n][::]), 2)  # NOTE: Endianess.
    q1 = int("".join(str(s) for s in bitstring[n:][::]), 2)  # NOTE: Endianess.
    if q0 in zero_bitstrings:
        assert q1 in zero_bitstrings
    elif q0 in one_bitstrings:
        assert q1 in one_bitstrings
    else:
        raise AssertionError("Neither 0_L 0_L nor 1_L 1_L were measured.")

In [78]:
to_run.count_ops()

OrderedDict([('sx', 213),
             ('rz', 131),
             ('cz', 117),
             ('measure', 14),
             ('x', 6),
             ('barrier', 2)])

In [79]:
nshots

10000

In [80]:
job = sampler.run(
    [to_run],
    shots=nshots,
)

In [ ]:
all_counts = [result.data.measure.get_counts() for result in job.result()]

In [ ]:
ev = get_lst_ev(all_counts[0], tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev)

## With DD

In [ ]:
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"

In [ ]:
job_dd = sampler.run(
    [compiled],
    shots=nshots,
)

In [ ]:
all_counts_dd = [result.data.measure.get_counts() for result in job_dd.result()]

In [ ]:
ev_dd = get_lst_ev(all_counts_dd[0], tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev_dd)

## With REM + DD

In [ ]:
from qiskit_experiments.library import LocalReadoutError

In [ ]:
experiment = LocalReadoutError(layout[n])

In [ ]:
result = experiment.run(computer)

In [ ]:
mitigator = result.analysis_results("Local Readout Mitigator").value

In [ ]:
# for i, qubit in enumerate(mitigator.qubits):
#     print(f"Qubit: {qubit}")
#     print(mitigator.mitigation_matrix(qubits=i))

In [ ]:
counts = all_counts[0]
mitigated_quasi_probs = mitigator.quasi_probabilities(counts)
mitigated_stddev = mitigated_quasi_probs._stddev_upper_bound
mitigated_probs = (mitigated_quasi_probs.nearest_probability_distribution().binary_probabilities())

In [ ]:
mitigated_counts = {k: round(v * nshots) for k, v in mitigated_probs.items()}

In [ ]:
sum(mitigated_counts.values())

In [ ]:
ev_rem = get_lst_ev(mitigated_counts, tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev_rem)

In [ ]:
counts_dd = all_counts_dd[0]
mitigated_quasi_probs = mitigator.quasi_probabilities(counts_dd)
mitigated_stddev = mitigated_quasi_probs._stddev_upper_bound
mitigated_probs = (mitigated_quasi_probs.nearest_probability_distribution().binary_probabilities())
mitigated_counts_dd = {k: round(v * nshots) for k, v in mitigated_probs.items()}

In [ ]:
sum(mitigated_counts_dd.values())

In [ ]:
ev_rem_dd = get_lst_ev(mitigated_counts_dd, tqdm(observable_elements), tqdm(stabilizer_elements))
print(ev_rem_dd)

## Save results

In [ ]:
save_key = f"bell_steane_code_zz_{computer.name}_{time_key}_job_raw_id_{job_physical.job_id()}_job_encoded_id_{job.job_id()}_job_encoded_dd_id_{job_dd.job_id()}_rem_calibration_id_{result.job_ids[0]}"

In [ ]:
import os

In [ ]:
os.mkdir(save_key)

In [ ]:
np.savetxt(f"{save_key}/qubits_physical.txt", layout[1])
np.savetxt(f"{save_key}/qubits_encoded.txt", layout[n])

In [ ]:
np.savetxt(f"{save_key}/nshots.txt", [nshots])

In [ ]:
np.savetxt(f"{save_key}/physical.txt", [ev_physical])
np.savetxt(f"{save_key}/encoded.txt", [ev])
np.savetxt(f"{save_key}/encoded_dd.txt", [ev_dd])
np.savetxt(f"{save_key}/encoded_rem.txt", [ev_rem])
np.savetxt(f"{save_key}/encoded_dd_rem.txt", [ev_rem_dd])